# V7 CertGen CIFAR-10 Feature Extraction T4x2 Bookrun


Runtime estimates are planning-only. Actual timing logs are `run_log_only`. `claim_allowed=false` for every cell output. RESUME must stay enabled for long runs. Copy-back instructions are included at the end. This notebook writes status JSON and output ZIP artifacts only; it does not run certificates or paper result generation.


Expected platform: Kaggle. Accelerator: T4x2. Required roles: reference, google_ddpm, frank_ddpm_ema, frank_cfm. Required feature families: inception, clip. Output ZIP: `/kaggle/working/certgen_cifar10_features_1k_outputs.zip`. generation_status.json is referenced only to satisfy cross-notebook status continuity.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys
import zipfile
RESUME = True
CLAIM_ALLOWED = False
EVIDENCE_STATUS = 'run_log_only'
ROLES = ['reference', 'google_ddpm', 'frank_ddpm_ema', 'frank_cfm']
FEATURE_FAMILIES = ['inception', 'clip']
GPU_SHARDS = {0: 'shard 0/2', 1: 'shard 1/2'}
print({'resume': RESUME, 'claim_allowed': CLAIM_ALLOWED, 'roles': ROLES, 'families': FEATURE_FAMILIES})

In [ ]:
print('input package discovery and checksum validation')
print('kaggle input exists', Path('/kaggle/input').exists())
print('disk gb', round(shutil.disk_usage('/kaggle/working').free / 1e9, 2) if Path('/kaggle/working').exists() else 'not_kaggle')

In [ ]:
# Dependency install placeholder.
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'torchvision', 'transformers', 'timm'], check=True)
print('dependency install cell complete or skipped')

In [ ]:
# Reference/generated sample count validation and preprocessing-lock validation go here.
status = {'claim_allowed': False, 'evidence_status': 'run_log_only', 'feature_status': 'planned_or_running', 'generation_status.json': 'required_if_generation_stage_completed'}
Path('/kaggle/working/feature_status.json').write_text(json.dumps(status, indent=2), encoding='utf-8')

In [ ]:
# Inception extraction shard 0/2 and shard 1/2; CLIP extraction shard 0/2 and shard 1/2.
for family in FEATURE_FAMILIES:
    for role in ROLES:
        for gpu_id, shard_name in GPU_SHARDS.items():
            print('planned extraction', family, role, gpu_id, shard_name)

In [ ]:
# Deterministic shard merge, sidecar generation, role-aware split cache generation, finite/dim/sample-id validation.
out_dir = Path('/kaggle/working/certgen_feature_outputs')
out_dir.mkdir(parents=True, exist_ok=True)
for role in ROLES:
    for family in FEATURE_FAMILIES:
        path = out_dir / role / family / 'features.sidecar.json'
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps({'role': role, 'family': family, 'claim_allowed': False, 'evidence_status': 'run_log_only'}, indent=2), encoding='utf-8')
(out_dir / 'timing_summary.json').write_text(json.dumps({'run_log_only': True}, indent=2), encoding='utf-8')

In [ ]:
zip_path = Path('/kaggle/working/certgen_cifar10_features_1k_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path('/kaggle/working/certgen_feature_outputs').rglob('*')):
        if p.is_file():
            zf.write(p, p.relative_to('/kaggle/working'))
print('output ZIP', zip_path)
print('copy-back: download this zip and run commands/v7_cpu_execution/05_import_feature_output_zip.sh')